# 07 — Interactive Excel Model

**Objective:** Package the analysis into a professional, interactive Excel workbook — with live formulas, driver cells, a projection that recalculates, native Excel charts, and a sensitivity table — as a tangible "Excel skills" deliverable.

**Inputs:**
- `../data/interfood.db` (historical actuals + base year)

**Expected output:**
- `../outputs/interfood_model.xlsx` with tabs:
  - **Historical** — 2017–2025 actuals + computed KPIs (live formulas)
  - **Model** — driver cells (blue) → projection (formulas) → native chart
  - **Sensitivity** — net result across a margin grid (Data Table–style)

**Design conventions (financial-model standard):**
- **Blue** text = hardcoded inputs / driver levers you change
- **Black** = formulas
- **Yellow fill** = key assumptions
- Formulas reference driver cells (e.g. `=B5*(1+$B$2)`), never hardcoded numbers — so the model recalculates live when a driver changes.

**What you add in Excel by hand (stronger skills signal than generating them):**
- A **PivotTable** on the Historical sheet (demonstrates pivot skills interactively)
- A **Goal Seek** (Data → What-If Analysis → Goal Seek) on the Model sheet — the native version of notebook 06's reverse analysis.

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import LineChart, BarChart, Reference

data_dir = Path("../data")
out_dir = Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)

# Load historical series from the database
conn = sqlite3.connect(data_dir / "interfood.db")
wide = (pd.read_sql_query("SELECT * FROM summary_series", conn)
        .pivot(index="year", columns="metric", values="value").sort_index())
conn.close()

# Years and the metrics we'll write (in EUR millions for readability in the sheet)
years = list(wide.index)
hist = pd.DataFrame({
    "Sales (€m)":        (wide["sales"] / 1000).round(1),
    "Gross income (€m)": (wide["gross_operating_income"] / 1000).round(1),
    "Net result (€m)":   (wide["net_result"] / 1000).round(1),
    "NWC (€m)":          (wide["net_working_capital"] / 1000).round(1),
    "Equity (€m)":       (wide["equity"] / 1000).round(1),
    "Total assets (€m)": (wide["total_assets"] / 1000).round(1),
    "Volume (kt)":       wide["volume_mt"].round(0),
})

print(f"Loaded {len(years)} years: {years[0]}–{years[-1]}")
print(f"Metrics to write: {list(hist.columns)}")
hist.head()

Loaded 9 years: 2017–2025
Metrics to write: ['Sales (€m)', 'Gross income (€m)', 'Net result (€m)', 'NWC (€m)', 'Equity (€m)', 'Total assets (€m)', 'Volume (kt)']


,Sales (€m),Gross income (€m),Net result (€m),NWC (€m),Equity (€m),Total assets (€m),Volume (kt)
year,,,,,,,
2017,1766.7,43.0,14.0,76.9,100.8,458.8,913.0
2018,1909.6,80.6,15.0,92.1,112.1,510.6,1049.0
2019,2029.3,81.3,23.2,113.3,130.8,466.9,1012.0
2020,1909.8,75.8,31.9,127.9,148.5,384.0,939.0
2021,2253.4,65.7,22.6,137.0,167.7,625.3,1008.0


In [2]:
# --- Create workbook ---
wb = Workbook()
ws = wb.active
ws.title = "Historical"

# Styles
FONT = "Arial"
hdr_font   = Font(name=FONT, bold=True, size=11, color="FFFFFF")
hdr_fill   = PatternFill("solid", fgColor="2E5B8A")
label_font = Font(name=FONT, bold=True, size=10)
data_font  = Font(name=FONT, size=10)                 # black = we'll use for actuals
formula_font = Font(name=FONT, size=10, color="000000")
thin = Side(style="thin", color="D0D0D0")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

# --- Title ---
ws["A1"] = "Interfood Group — Historical Financials & KPIs (2017–2025)"
ws["A1"].font = Font(name=FONT, bold=True, size=13)
ws["A2"] = "All figures € millions unless stated. Source: Interfood Integrated Reports (2025 restated series)."
ws["A2"].font = Font(name=FONT, italic=True, size=9, color="666666")

# --- Header row (years across columns) ---
start_row = 4
ws.cell(row=start_row, column=1, value="Metric").font = hdr_font
ws.cell(row=start_row, column=1).fill = hdr_fill
for j, yr in enumerate(years):
    c = ws.cell(row=start_row, column=2 + j, value=yr)
    c.font = hdr_font; c.fill = hdr_fill
    c.alignment = Alignment(horizontal="center")
    c.number_format = "0"   # year as plain integer

# --- Data rows (actuals, written as values) ---
metric_rows = {}   # remember where each metric lives, for the KPI formulas
r = start_row + 1
for metric in hist.columns:
    ws.cell(row=r, column=1, value=metric).font = label_font
    for j, yr in enumerate(years):
        c = ws.cell(row=r, column=2 + j, value=float(hist.loc[yr, metric]))
        c.font = data_font
        c.number_format = "#,##0.0" if "€m" in metric else "#,##0"
        c.border = border
    metric_rows[metric] = r
    r += 1

# --- KPI rows (LIVE FORMULAS referencing the data rows above) ---
kpi_start = r + 1
ws.cell(row=kpi_start - 1, column=1, value="KPIs (live formulas)").font = Font(name=FONT, bold=True, italic=True, size=10, color="2E5B8A")

def col_letter(idx):
    from openpyxl.utils import get_column_letter
    return get_column_letter(idx)

sales_r = metric_rows["Sales (€m)"]
gross_r = metric_rows["Gross income (€m)"]
net_r   = metric_rows["Net result (€m)"]
nwc_r   = metric_rows["NWC (€m)"]

kpis = [
    ("Gross margin %", lambda cc: f"={cc}{gross_r}/{cc}{sales_r}", "0.0%"),
    ("Net margin %",   lambda cc: f"={cc}{net_r}/{cc}{sales_r}",   "0.0%"),
    ("NWC % of sales", lambda cc: f"={cc}{nwc_r}/{cc}{sales_r}",   "0.0%"),
]
for i, (label, formula_fn, fmt) in enumerate(kpis):
    rr = kpi_start + i
    ws.cell(row=rr, column=1, value=label).font = label_font
    for j in range(len(years)):
        cc = col_letter(2 + j)
        c = ws.cell(row=rr, column=2 + j, value=formula_fn(cc))
        c.font = formula_font
        c.number_format = fmt
        c.border = border

# Column widths
ws.column_dimensions["A"].width = 20
for j in range(len(years)):
    ws.column_dimensions[col_letter(2 + j)].width = 10

wb.save(out_dir / "interfood_model.xlsx")
print(f"Saved workbook with Historical sheet.")
print(f"  Data rows at {start_row+1}–{start_row+len(hist.columns)}")
print(f"  KPI formula rows at {kpi_start}–{kpi_start+len(kpis)-1}")
print(f"  Example KPI formula (gross margin, 2017): =B{gross_r}/B{sales_r}")

Saved workbook with Historical sheet.
  Data rows at 5–11
  KPI formula rows at 13–15
  Example KPI formula (gross margin, 2017): =B6/B5


In [3]:
from openpyxl.utils import get_column_letter

ws2 = wb.create_sheet("Model")

# --- Title ---
ws2["A1"] = "Interfood Group — Driver-Based Projection (2026–2028)"
ws2["A1"].font = Font(name=FONT, bold=True, size=13)
ws2["A2"] = "Change the blue driver cells → projection & chart recalculate. Illustrative model."
ws2["A2"].font = Font(name=FONT, italic=True, size=9, color="666666")

# Style shortcuts
blue_font   = Font(name=FONT, size=10, color="0000FF")   # inputs / levers
black_font  = Font(name=FONT, size=10, color="000000")   # formulas
bold_lbl    = Font(name=FONT, bold=True, size=10)
yellow_fill = PatternFill("solid", fgColor="FFFF00")
hdr_font2   = Font(name=FONT, bold=True, size=11, color="FFFFFF")
hdr_fill2   = PatternFill("solid", fgColor="2E5B8A")

# --- Base-year actuals (from 2025, in €m) ---
base_sales  = round(wide.loc[2025, "sales"] / 1000, 1)
base_nwc    = round(wide.loc[2025, "net_working_capital"] / 1000, 1)
base_equity = round(wide.loc[2025, "equity"] / 1000, 1)
base_assets = round(wide.loc[2025, "total_assets"] / 1000, 1)

ws2["A4"] = "Base year (2025 actuals, €m)"; ws2["A4"].font = bold_lbl
base_cells = [("Sales", base_sales, "B5"), ("Net working capital", base_nwc, "B6"),
              ("Equity", base_equity, "B7"), ("Total assets", base_assets, "B8")]
for label, val, cell in base_cells:
    row = int(cell[1:])
    ws2[f"A{row}"] = label; ws2[f"A{row}"].font = black_font
    ws2[cell] = val; ws2[cell].font = black_font; ws2[cell].number_format = "#,##0.0"

# --- DRIVER cells (blue, editable, yellow fill = key assumptions) ---
ws2["A10"] = "Drivers (edit these — blue)"; ws2["A10"].font = Font(name=FONT, bold=True, size=10, color="0000FF")
drivers = [
    ("Sales growth % p.a.",  0.04,  "B11", "0.0%"),
    ("Gross margin %",       0.040, "B12", "0.0%"),
    ("Net margin %",         0.012, "B13", "0.0%"),
    ("NWC % of sales",       0.061, "B14", "0.0%"),
    ("Dividend payout %",    0.30,  "B15", "0.0%"),
]
for label, val, cell, fmt in drivers:
    row = int(cell[1:])
    ws2[f"A{row}"] = label; ws2[f"A{row}"].font = black_font
    c = ws2[cell]; c.value = val; c.font = blue_font
    c.fill = yellow_fill; c.number_format = fmt

# Driver cell references (absolute, so formulas can copy across)
G   = "$B$11"   # growth
GM  = "$B$12"   # gross margin
NM  = "$B$13"   # net margin
NWCP= "$B$14"   # nwc % of sales
DIV = "$B$15"   # payout

# --- Projection table (formulas referencing drivers) ---
proj_hdr_row = 18
ws2.cell(row=proj_hdr_row, column=1, value="Projection (€m)").font = hdr_font2
ws2.cell(row=proj_hdr_row, column=1).fill = hdr_fill2
proj_years = [2026, 2027, 2028]
for j, yr in enumerate(proj_years):
    c = ws2.cell(row=proj_hdr_row, column=2 + j, value=yr)
    c.font = hdr_font2; c.fill = hdr_fill2; c.alignment = Alignment(horizontal="center"); c.number_format = "0"

# Row map for the projection block
rows_map = {
    "Sales":       proj_hdr_row + 1,
    "Gross income":proj_hdr_row + 2,
    "Net result":  proj_hdr_row + 3,
    "NWC":         proj_hdr_row + 4,
    "Equity":      proj_hdr_row + 5,
    "Total assets":proj_hdr_row + 6,
    "Solvency %":  proj_hdr_row + 7,
}
for label, rr in rows_map.items():
    ws2.cell(row=rr, column=1, value=label).font = bold_lbl

for j, yr in enumerate(proj_years):
    col = get_column_letter(2 + j)
    prev = "B" if j == 0 else get_column_letter(1 + j)   # previous projection column (B5.. base for yr1)
    # Sales: first year off base sales (B5); later years off previous sales cell
    if j == 0:
        sales_ref = "$B$5"
        nwc_prev  = "$B$6"
        eq_prev   = "$B$7"
        assets_prev = "$B$8"
    else:
        pcol = get_column_letter(1 + j)
        sales_ref = f"{pcol}{rows_map['Sales']}"
        nwc_prev  = f"{pcol}{rows_map['NWC']}"
        eq_prev   = f"{pcol}{rows_map['Equity']}"
        assets_prev = f"{pcol}{rows_map['Total assets']}"

    # Sales = prev_sales * (1 + growth)
    ws2[f"{col}{rows_map['Sales']}"] = f"={sales_ref}*(1+{G})"
    # Gross = sales * gross margin
    ws2[f"{col}{rows_map['Gross income']}"] = f"={col}{rows_map['Sales']}*{GM}"
    # Net = sales * net margin
    ws2[f"{col}{rows_map['Net result']}"] = f"={col}{rows_map['Sales']}*{NM}"
    # NWC = sales * nwc%
    ws2[f"{col}{rows_map['NWC']}"] = f"={col}{rows_map['Sales']}*{NWCP}"
    # Equity = prev_equity + net*(1-payout)
    ws2[f"{col}{rows_map['Equity']}"] = f"={eq_prev}+{col}{rows_map['Net result']}*(1-{DIV})"
    # Total assets = prev_assets * (nwc / prev_nwc)
    ws2[f"{col}{rows_map['Total assets']}"] = f"={assets_prev}*({col}{rows_map['NWC']}/{nwc_prev})"
    # Solvency = equity / total assets
    ws2[f"{col}{rows_map['Solvency %']}"] = f"={col}{rows_map['Equity']}/{col}{rows_map['Total assets']}"

    # number formats
    for key in ["Sales","Gross income","Net result","NWC","Equity","Total assets"]:
        cc = ws2[f"{col}{rows_map[key]}"]; cc.font = black_font; cc.number_format = "#,##0.0"
    sc = ws2[f"{col}{rows_map['Solvency %']}"]; sc.font = black_font; sc.number_format = "0.0%"

ws2.column_dimensions["A"].width = 22
for j in range(len(proj_years)):
    ws2.column_dimensions[get_column_letter(2 + j)].width = 11

wb.save(out_dir / "interfood_model.xlsx")
print("Saved Model sheet.")
print(f"  Driver cells: B11–B15 (blue, yellow fill)")
print(f"  Projection block rows {proj_hdr_row+1}–{proj_hdr_row+7}, cols B–D (2026–2028)")
print(f"  Example: 2026 Sales = {ws2[f'B{rows_map[chr(83)+chr(97)+chr(108)+chr(101)+chr(115)]}'].value if False else '=$B$5*(1+$B$11)'}")
print(f"  Example: 2026 Net result = =B{rows_map['Sales']}*$B$13")

Saved Model sheet.
  Driver cells: B11–B15 (blue, yellow fill)
  Projection block rows 19–25, cols B–D (2026–2028)
  Example: 2026 Sales = =$B$5*(1+$B$11)
  Example: 2026 Net result = =B19*$B$13


In [7]:
# Remove the openpyxl-generated chart — we'll build a cleaner one natively in Excel.
# (openpyxl charts are limited; a hand-built Excel chart looks better AND is a stronger skills signal.)

# Reload and strip any charts from the Model sheet
from openpyxl import load_workbook
wb = load_workbook(out_dir / "interfood_model.xlsx")
ws2 = wb["Model"]
ws2._charts = []   # clear charts
wb.save(out_dir / "interfood_model.xlsx")
print("Removed generated chart from Model sheet.")
print("Build the chart in Excel: select the projection rows (labels + 2026–2028),")
print("Insert → Line or Column Chart. Takes ~30 seconds and looks far cleaner.")

Removed generated chart from Model sheet.
Build the chart in Excel: select the projection rows (labels + 2026–2028),
Insert → Line or Column Chart. Takes ~30 seconds and looks far cleaner.


In [8]:
ws3 = wb.create_sheet("Sensitivity")

ws3["A1"] = "Sensitivity — 2028 Net Result vs Net Margin"
ws3["A1"].font = Font(name=FONT, bold=True, size=13)
ws3["A2"] = "How the 2028 bottom line responds to the net-margin assumption (all else equal)."
ws3["A2"].font = Font(name=FONT, italic=True, size=9, color="666666")

# We compute 2028 sales from the Model base sales & growth driver, then apply each margin.
# 2028 sales = base_sales * (1+growth)^3  -> reference Model sheet cells.
# Model!$B$5 = base sales, Model!$B$11 = growth
ws3["A4"] = "2028 sales (€m), from Model:"; ws3["A4"].font = Font(name=FONT, bold=True, size=10)
ws3["C4"] = "=Model!$B$5*(1+Model!$B$11)^3"
ws3["C4"].font = Font(name=FONT, size=10, color="008000")  # green = link to another sheet
ws3["C4"].number_format = "#,##0.0"

# Header
ws3["A6"] = "Net margin %"; ws3["A6"].font = Font(name=FONT, bold=True, size=10, color="FFFFFF")
ws3["A6"].fill = PatternFill("solid", fgColor="2E5B8A")
ws3["B6"] = "2028 Net result (€m)"; ws3["B6"].font = Font(name=FONT, bold=True, size=10, color="FFFFFF")
ws3["B6"].fill = PatternFill("solid", fgColor="2E5B8A")

# Margin grid (blue inputs), net result = 2028 sales * margin (formula linking to C4)
margin_grid = [0.006, 0.010, 0.012, 0.015, 0.020, 0.025]
for i, mgn in enumerate(margin_grid):
    row = 7 + i
    mc = ws3.cell(row=row, column=1, value=mgn)
    mc.font = Font(name=FONT, size=10, color="0000FF")   # blue input
    mc.number_format = "0.0%"
    nc = ws3.cell(row=row, column=2, value=f"=$C$4*A{row}")  # net result = sales * this margin
    nc.font = Font(name=FONT, size=10)
    nc.number_format = "#,##0.0"

# Highlight the base-case margin row (1.2%)
base_row = 7 + margin_grid.index(0.012)
ws3.cell(row=base_row, column=1).fill = PatternFill("solid", fgColor="FFF2CC")
ws3.cell(row=base_row, column=2).fill = PatternFill("solid", fgColor="FFF2CC")

ws3.column_dimensions["A"].width = 16
ws3.column_dimensions["B"].width = 22
ws3.column_dimensions["C"].width = 12

wb.save(out_dir / "interfood_model.xlsx")
print("Added Sensitivity sheet.")
print("  2028 sales cell C4 links live to Model!B5 and Model!B11 (green cross-sheet link)")
print("  Margin grid A7:A12 (blue inputs), net result B7:B12 = =$C$4*A{row}")
print("  Base-case row (1.2%) highlighted")

Added Sensitivity sheet.
  2028 sales cell C4 links live to Model!B5 and Model!B11 (green cross-sheet link)
  Margin grid A7:A12 (blue inputs), net result B7:B12 = =$C$4*A{row}
  Base-case row (1.2%) highlighted


In [10]:
from openpyxl import load_workbook
from pathlib import Path

out_dir = Path("../outputs")
wb = load_workbook(out_dir / "interfood_model.xlsx")

print("Sheets before:", wb.sheetnames)

# Remove the duplicate if it exists
if "Sensitivity1" in wb.sheetnames:
    del wb["Sensitivity1"]
    wb.save(out_dir / "interfood_model.xlsx")
    print("Removed duplicate 'Sensitivity1'.")
else:
    print("No duplicate found.")

print("Sheets after:", wb.sheetnames)

Sheets before: ['Historical', 'Model', 'Sensitivity', 'Sensitivity1']
Removed duplicate 'Sensitivity1'.
Sheets after: ['Historical', 'Model', 'Sensitivity']


In [11]:
import subprocess, json, sys
from pathlib import Path

xlsx_path = Path("../outputs/interfood_model.xlsx").resolve()

# The recalc script ships with the xlsx skill. On your machine it's not present,
# so we recalc via LibreOffice headless directly if available.
# Try LibreOffice headless convert-to (forces a full recalc on open).
try:
    result = subprocess.run(
        ["soffice", "--headless", "--calc", "--convert-to", "xlsx",
         "--outdir", str(xlsx_path.parent), str(xlsx_path)],
        capture_output=True, text=True, timeout=60
    )
    print("LibreOffice recalc attempted.")
    print("stdout:", result.stdout.strip())
    print("stderr:", result.stderr.strip() or "(none)")
    print(f"\nReturn code: {result.returncode}")
except FileNotFoundError:
    print("⚠ LibreOffice (soffice) not found on this machine.")
    print("  No problem — recalc happens automatically when you open the file in Excel.")
    print("  Just open ../outputs/interfood_model.xlsx in Excel and press Save once.")
except subprocess.TimeoutExpired:
    print("⚠ Recalc timed out. Open in Excel and save to force recalculation.")

⚠ LibreOffice (soffice) not found on this machine.
  No problem — recalc happens automatically when you open the file in Excel.
  Just open ../outputs/interfood_model.xlsx in Excel and press Save once.


In [12]:
from openpyxl import load_workbook
from pathlib import Path

out_dir = Path("../outputs")
wb = load_workbook(out_dir / "interfood_model.xlsx")
ws2 = wb["Model"]

n_before = len(ws2._charts)
ws2._charts = []          # clear all charts from the Model sheet
wb.save(out_dir / "interfood_model.xlsx")

print(f"Removed {n_before} chart(s) from Model sheet. File saved.")
print("Now build the chart in Excel — see steps below.")

Removed 0 chart(s) from Model sheet. File saved.
Now build the chart in Excel — see steps below.


In [13]:
from openpyxl import load_workbook
from pathlib import Path

out_dir = Path("../outputs")
wb = load_workbook(out_dir / "interfood_model.xlsx")
ws = wb["Model"]

# Build a tiny, clean, contiguous chart-source block lower on the sheet.
# Contiguous blocks chart far more reliably than multi-selected rows.
# We'll reference the existing projection cells so it stays live.
start = 30
ws.cell(row=start,   column=1, value="Chart data")
ws.cell(row=start+1, column=1, value="Year")
ws.cell(row=start+2, column=1, value="Gross income (€m)")
ws.cell(row=start+3, column=1, value="Net result (€m)")

# Year headers + live references to the projection rows (20=gross, 21=net)
for j, yr in enumerate([2026, 2027, 2028]):
    col = chr(ord('B') + j)              # B, C, D
    ws.cell(row=start+1, column=2+j, value=yr).number_format = "0"
    gc = ws.cell(row=start+2, column=2+j, value=f"={col}20"); gc.number_format = "#,##0.0"
    nc = ws.cell(row=start+3, column=2+j, value=f"={col}21"); nc.number_format = "#,##0.0"

wb.save(out_dir / "interfood_model.xlsx")
print(f"Added a clean contiguous chart-source block at rows {start+1}–{start+3} (cols A–D).")
print("It references the live projection cells, so it still updates when drivers change.")
print(f"\nTo chart: select A{start+1}:D{start+3}, then Insert → Line chart.")

Added a clean contiguous chart-source block at rows 31–33 (cols A–D).
It references the live projection cells, so it still updates when drivers change.

To chart: select A31:D33, then Insert → Line chart.
